<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026word2vec_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# word2vec (2026) 実習

<div align='right'>
<a href='mailto:asakawa@ieee.org'>Shin Aasakawa</a>, all rights reserved.<br/>
Date: 02/Oct/2026<br/>
Original Date: 02/Oct/2026<br/>
MIT license
</div>



<center>

<img src="https://komazawa-deep-learning.github.io/assets/2013Mikolov_KingQueenFig.svg" width="600"><br/>
From Mikolov et. al (2013) Linguistic Regularities in Continuous SpaceWord Representations, Fig. 2<br/>
</center>
全ての概念がある多次元空間内に存在するとします。各概念はこの空間内での位置ベクトルとして表現されます。
このとき，高等学校のときのベクトルの知識を思い出すと，$\mathbf{a}-\mathbf{b}$ は，ベクトル $\mathbf{b}$ から ベクトル $\mathbf{a}$ へ向かうベクトルとして表されます。
すなわち女王ベクトルから王様ベクトルを引くということは，王様を原点と考えて，女王を眺めた方向を表すベクトルと考えることができます。
このような，眺める方向を表すベクトルと，別のベクトル 女性ベクトルから男性ベクトルを引いたベクトル，
すなわち，男性を原点として女性を眺めたベクトルを考えます。
このベクトル 2 つ，<b><font color="teal">女王 - 王様</font></b> と <b><font color="teal">女 - 男</font></b> がほぼ重なることを
比喩的に <b><font color="teal">女王 - 王様 = 女 - 男</font></b> となります。この式を少し変形すれば，
<b><font color="teal">王様 - 男 + 女 = 女王</font></b> となるでしょう。これを言葉で表せば，
<b><font color="red">王様から男を引いて，女を足すと女王になる</font></b>，という演算が形式的に成り立ちます。

<br/>
<br/>
<center>
<img src="https://komazawa-deep-learning.github.io/assets/2013Mikolov_FigCountries.svg" width="680"><br/>
Modified from Mikolv et. al (2013) Distributed Representations of Words and Phrases and their Compositionality, Fig. 2</br/>

<br/><br/>
<img src="https://komazawa-deep-learning.github.io/assets/2013Mikolov_Fig1.svg" width="840"><br/>
</center>


In [ ]:
!pip install -q -U gensim

import gensim, numpy
print("gensim:", gensim.__version__)
print("numpy:", numpy.__version__)

# ランタイムを強制終了して再起動させる(Colabは自動再接続する)
import os
os.kill(os.getpid(), 9)

In [ ]:
# 2025 年 05 月 01 日時点でのメモ
# numpy のバージョンを強制的に 1.26.4 にダウングレードした場合，ランタイムの再起動をしないと
# ダウングレードした numpy が実行時に反映されない。
# このため一度このセルを実行した後に,上のメニューバー左から 5 つ目の「ランタイム」から「セッションを再起動する」
# を選択して再度このセルを実行する必要がある。

# Google Colab 上で実行しているかどうかを判定
import IPython
isColab = 'google.colab' in str(IPython.get_ipython())

# Google colab で実行している場合必要なライブラリをインストールして word2vec ファイルをダウンロードする
if isColab:
    !pip install pydrive2

    #Import modules
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive
    from google.colab import auth
    from oauth2client.client import GoogleCredentials

    #Authenticate and create the PyDrive client
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    #Get the Shareable link
    # 2017Jul_jawiki-wakati_neologd_hid200_win20_neg20_cbow.bin.gz: 1Rp3HbDkbpzMg5ehq1ARwCATX8iZAxTgj
    # 2017Jul_jawiki-wakati_neologd_hid200_win20_neg20_sgns.bin.gz: 19BKVOBNHESt1K8725UTM9J3OpqK7YlVb

    # 2021_05jawiki_hid200_win20_neg20_sgns.bin.gz: 1JTkU5SUBU2GkURCYeHkAWYs_Zlbqob0s
    # 2021_05jawiki_hid200_win20_neg20_cbow.bin.gz: 1VPL2Mr9JgWHik9HjRmcADoxXIdrQ3ds7
    # 2021_05jawiki_hid128_win10_neg10_sgns.bin.gz: 1OWmFOVRC6amCxsomcRwdA6ILAA5s4y4M
    # 2021_05jawiki_hid128_win10_neg10_cbow.bin.gz: 1B9HGhLZOja4Xku5c_d-kMhCXn1LBZgDb

    downloaded = drive.CreateFile({'id':"1Rp3HbDkbpzMg5ehq1ARwCATX8iZAxTgj"})
    downloaded.GetContentFile('2017Jul_jawiki-wakati_neologd_hid200_win20_neg20_cbow.bin.gz')
    # 上記は cbow で訓練済ファイルをダウンロードしています。
    # skip gram モデルで訓練済モデルに変更する場合には，上記情報に従って適宜書き換えてください。

In [ ]:
# 上セルでダウンロードした訓練済 word2vec ファイルを読み込む
word2vec_file='2017Jul_jawiki-wakati_neologd_hid200_win20_neg20_cbow.bin.gz'
model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_file,
                                                        encoding='utf-8',
                                                        unicode_errors='replace',
                                                        binary=True)

In [ ]:
model.most_similar(positive=['woman', 'king'], negative=['man'], topn=5)

In [ ]:
model.most_similar(positive=['woman', 'king'], negative=['man'], topn=5)

In [ ]:
# model.most_similar(positive=['日本心理学会'], negative=['心理学'], topn=10)
# model.most_similar(positive=['心理学'], negative=['科学'], topn=10)
model.most_similar(positive=['心理学'], negative=['実験'], topn=10)

In [ ]:
model.most_similar(positive=['ユニクロ','錦織圭'], negative=['テニス'], topn=10)